# nanochat GPT-2 Pretraining — TPU v5e-8 (torch XLA SPMD)

Full-speed rebuild of **[karpathy/nanochat](https://github.com/karpathy/nanochat)** (latest, Feb 2026) \
targeting the **GPT-2 124M class model** on a single TPU v5e-8 pod.

**Architecture (nanochat latest)**
- RoPE rotary embeddings · QK-normalization · ReLU² MLP  
- Per-layer residual scalars λ_resid / λ_x0 · Logit soft-cap 30·tanh(·/30)  
- Untied token + output embeddings · Zero-init output projections  

**Optimizer (nanochat latest)**
- **Muon** (Newton-Schulz orthogonalization) for all weight matrices  
- **AdamW** β=(0.8, 0.95) for embeddings & scalars  
- Trapezoidal LR schedule (0 warmup, 40 % linear warmdown)  

**Scale (Chinchilla-optimal)**  
- 124 M compute params → **2.48 B training tokens** (20×N rule)  
- ~4 730 steps at batch = 2¹⁹ = 524 288 tokens  

**Estimated wall-clock on TPU v5e-8:** ~25-40 min  


In [3]:
# ── Install (run once, then restart kernel) ──────────────────────────────────
# On a fresh GCP TPU VM torch_xla may already be present; this is a no-op then.
!pip install -q torch==2.9.0 torch_xla[tpu]==2.9.0 \
    -f https://storage.googleapis.com/libtpu-releases/index.html \
    --no-warn-script-location

!pip install -q tiktoken "datasets>=3.0" tqdm

print("Done. Restart the kernel now, then run cells 3 onward.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.8.0+cpu requires torch==2.8.0, but you have torch 2.9.0 which is incompatible.
torchvision 0.23.0+cpu requires torch==2.8.0, but you have torch 2.9.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Done. Restart the kernel now, then run cells 3 onward.


In [1]:
import os, math, time, json, threading, queue
from typing import Optional, Iterator
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from datasets import load_dataset

In [2]:
import os, math, time, json, threading, queue
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
print("torch:", torch.__version__)

torch: 2.9.0+cu128


In [3]:
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
print("torch_xla:", torch_xla.__version__)

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:246: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


torch_xla: 2.9.0


In [4]:
os.environ['PJRT_DEVICE'] = 'TPU'
print(xr.global_runtime_device_count())

8


E0000 00:00:1773061985.673764     108 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


In [5]:
import torch_xla.distributed.spmd as xs
xr.use_spmd()
print("SPMD ok")

SPMD ok


In [7]:
import numpy as np
NUM_DEVICES = xr.global_runtime_device_count()
print("devices:", NUM_DEVICES)

import torch_xla.distributed.spmd as xs
MESH = xs.Mesh(np.arange(NUM_DEVICES), (NUM_DEVICES,), ('data',))
print("Mesh:", MESH)

devices: 8
Mesh: {'device_ids': [0, 1, 2, 3, 4, 5, 6, 7], 'mesh_shape': (8,), 'axis_names': ('data',)}


In [8]:
# ═══════════════════════════════════════════════════════════════
# MODEL HYPERPARAMETERS
# nanochat 'depth' dial → all other dims derived automatically
# ═══════════════════════════════════════════════════════════════
DEPTH      = 12       # nanochat depth dial (12 → GPT-2 class)
ASPECT     = 64       # nanochat constant: n_embd = depth × aspect
HEAD_DIM   = 64       # fixed head dim (nanochat)

N_LAYER    = DEPTH
N_EMBD     = DEPTH * ASPECT          # 768
N_HEAD     = N_EMBD  // HEAD_DIM     # 12
N_KV_HEAD  = N_HEAD                  # 12  (no KV compression at this scale)
SEQ_LEN    = 1024                    # context window

# Vocab: tiktoken r50k_base (GPT-2) = 50 257 tokens
# Padded to nearest multiple of 128 for TPU tensor-core efficiency
VOCAB_RAW   = 50_257
VOCAB_SIZE  = ((VOCAB_RAW + 127) // 128) * 128   # 50 304

# ═══════════════════════════════════════════════════════════════
# TRAINING BUDGET  (Chinchilla-optimal: 20 × compute-params)
# ═══════════════════════════════════════════════════════════════
TARGET_COMPUTE_PARAMS = 124_000_000   # GPT-2 base reference
TARGET_TOKENS         = 20 * TARGET_COMPUTE_PARAMS   # 2.48 B

# ── Batch ──────────────────────────────────────────────────────

TOTAL_BATCH_TOKENS = 2 ** 16   # 65536  (was 2**19)
TOTAL_SEQS         = TOTAL_BATCH_TOKENS // SEQ_LEN   # 64
PER_CHIP_SEQS      = TOTAL_SEQS // NUM_DEVICES        # 8
NUM_STEPS          = math.ceil(TARGET_TOKENS / TOTAL_BATCH_TOKENS)  # ~37840

# ── LR: nanochat defaults, scaled by (n_embd/768)^-0.5 ────────
LR_SCALE      = (N_EMBD / 768) ** -0.5    # = 1.0 at depth 12
MATRIX_LR     = 0.02                       # Muon
EMBED_LR      = 0.200 * LR_SCALE           # AdamW for wte
LMHEAD_LR     = 0.004 * LR_SCALE           # AdamW for lm_head
SCALAR_LR     = 0.005                      # AdamW for resid/x0 lambdas
X0_LR         = 0.5                        # AdamW for x0_lambdas (higher)

ADAM_BETAS    = (0.8, 0.95)
ADAM_EPS      = 1e-10
MUON_MOMENTUM = 0.95
MUON_NS_STEPS = 5

# Weight decay scales ∝ 1/depth² (nanochat: tuned at d12, = 0.2)
WEIGHT_DECAY  = 0.2 * (12 / DEPTH) ** 2    # = 0.2 at depth 12

# ── LR schedule: no warmup, 40 % linear warmdown to 0 ──────────
WARMDOWN_START = int(NUM_STEPS * 0.6)   # begin warmdown at 60 %

# ── Logging ────────────────────────────────────────────────────
LOG_EVERY  = 20
VAL_EVERY  = 200
CKPT_EVERY = 1000
CKPT_DIR   = './checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

USE_WANDB  = False   # flip to True to enable wandb logging

print(f'n_layer={N_LAYER}  n_embd={N_EMBD}  n_head={N_HEAD}')
print(f'seq_len={SEQ_LEN}  vocab={VOCAB_SIZE}')
print(f'steps={NUM_STEPS:,}  batch={TOTAL_BATCH_TOKENS:,} tokens  per_chip={PER_CHIP_SEQS} seqs')
print(f'Training tokens: {TARGET_TOKENS/1e9:.2f} B')


n_layer=12  n_embd=768  n_head=12
seq_len=1024  vocab=50304
steps=37,842  batch=65,536 tokens  per_chip=8 seqs
Training tokens: 2.48 B


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  nanochat GPT Architecture  —  faithfully ported to XLA      ║
# ╚══════════════════════════════════════════════════════════════╝

from typing import Optional

# ── Utility: RMSNorm (no learnable weight, nanochat style) ──────────────────
def rms_norm(x: torch.Tensor) -> torch.Tensor:
    """Parameter-free RMSNorm using F.rms_norm (PyTorch ≥ 2.4)."""
    return F.rms_norm(x, (x.size(-1),))


# ── Rotary Positional Embeddings (RoPE) ─────────────────────────────────────
def precompute_rotary(seq_len: int, head_dim: int,
                      base: float = 10_000.0) -> tuple[torch.Tensor, torch.Tensor]:
    """Returns (cos, sin) buffers, shape (1, seq_len, 1, head_dim//2)."""
    half     = head_dim // 2
    theta    = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t        = torch.arange(seq_len, dtype=torch.float32)
    freqs    = torch.outer(t, theta)          # (T, half)
    cos      = freqs.cos()[None, :, None, :]  # (1, T, 1, half)
    sin      = freqs.sin()[None, :, None, :]
    return cos.to(torch.bfloat16), sin.to(torch.bfloat16)


def apply_rotary(x: torch.Tensor,
                 cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """x: (B, T, H, D);  cos/sin: (1, T, 1, D//2)"""
    half = x.size(-1) // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


# ── Causal Self-Attention (GQA-ready, SDPA backend) ─────────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, n_kv_head: int, head_dim: int):
        super().__init__()
        self.n_head    = n_head
        self.n_kv_head = n_kv_head
        self.head_dim  = head_dim
        self.n_rep     = n_head // n_kv_head   # GQA repetitions (=1 here)

        # Separate Q / K / V projections (no bias, nanochat style)
        self.c_q    = nn.Linear(n_embd, n_head    * head_dim, bias=False)
        self.c_k    = nn.Linear(n_embd, n_kv_head * head_dim, bias=False)
        self.c_v    = nn.Linear(n_embd, n_kv_head * head_dim, bias=False)
        self.c_proj = nn.Linear(n_head * head_dim, n_embd,    bias=False)

    def forward(self, x: torch.Tensor,
                cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        H, Hkv, D = self.n_head, self.n_kv_head, self.head_dim

        # (B, T, H*D)  →  (B, T, H, D)
        q = self.c_q(x).view(B, T, H,   D)
        k = self.c_k(x).view(B, T, Hkv, D)
        v = self.c_v(x).view(B, T, Hkv, D)

        # RoPE
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        # QK-normalization  (critical stabilizer in nanochat)
        q = rms_norm(q)
        k = rms_norm(k)

        # GQA expansion if n_rep > 1  (free no-op when n_rep == 1)
        if self.n_rep > 1:
            k = k.unsqueeze(3).expand(B, T, Hkv, self.n_rep, D).reshape(B, T, H, D)
            v = v.unsqueeze(3).expand(B, T, Hkv, self.n_rep, D).reshape(B, T, H, D)

        # SDPA expects (B, H, T, D) layout
        q = q.transpose(1, 2)   # (B, H, T, D)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # XLA-native SDPA — causal mask auto-applied
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # (B, H, T, D)  →  (B, T, H*D)  →  (B, T, n_embd)
        y = y.transpose(1, 2).contiguous().view(B, T, H * D)
        return self.c_proj(y)


# ── Feed-forward: Squared ReLU (nanochat) ────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, n_embd: int):
        super().__init__()
        self.c_fc   = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.c_proj(F.relu(self.c_fc(x)).square())   # ReLU²


# ── Transformer Block ─────────────────────────────────────────────────────────
# Parallel attn + mlp on the same pre-normed activations.
# (Reduces serial depth by 2×; used in modded-nanogpt and nanochat.)
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, n_kv_head: int, head_dim: int):
        super().__init__()
        self.attn = CausalSelfAttention(n_embd, n_head, n_kv_head, head_dim)
        self.mlp  = MLP(n_embd)

    def forward(self, x: torch.Tensor,
                cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        h = rms_norm(x)                              # single shared norm
        return self.attn(h, cos, sin) + self.mlp(h)  # parallel sub-layers


# ── Full GPT Model ────────────────────────────────────────────────────────────
class GPT(nn.Module):
    """nanochat GPT — XLA-optimised, no Flash Attention dependency."""

    def __init__(
        self,
        n_layer:    int = N_LAYER,
        n_embd:     int = N_EMBD,
        n_head:     int = N_HEAD,
        n_kv_head:  int = N_KV_HEAD,
        head_dim:   int = HEAD_DIM,
        seq_len:    int = SEQ_LEN,
        vocab_size: int = VOCAB_SIZE,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd     = n_embd
        self.n_layer    = n_layer

        # Token embedding  (untied from lm_head — nanochat style)
        self.wte = nn.Embedding(vocab_size, n_embd)

        # Transformer blocks
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, n_kv_head, head_dim) for _ in range(n_layer)]
        )

        # Per-layer residual scalars  (nanochat core feature)
        #   x_new = resid_lambda[i]*x + x0_lambda[i]*x0 + block(norm(x))
        self.resid_lambdas = nn.Parameter(torch.ones(n_layer))
        self.x0_lambdas    = nn.Parameter(torch.full((n_layer,), 0.1))

        # Unembedding  (separate weights — untied)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Precomputed rotary buffers (bfloat16, non-persistent)
        rc, rs = precompute_rotary(seq_len, head_dim)
        self.register_buffer('rot_cos', rc, persistent=False)
        self.register_buffer('rot_sin', rs, persistent=False)

        self._init_weights()

    # ── Weight initialisation (nanochat) ──────────────────────────────────────
    def _init_weights(self):
        """Uniform init for QKV/MLP_fc; zero for output projections."""
        s = (3 ** 0.5) * (self.n_embd ** -0.5)   # keeps same std as Normal(0,1/√n)
        nn.init.normal_(self.wte.weight,      std=1.0)
        nn.init.normal_(self.lm_head.weight,  std=0.001)
        for blk in self.blocks:
            for proj in (blk.attn.c_q, blk.attn.c_k, blk.attn.c_v):
                nn.init.uniform_(proj.weight, -s, s)
            nn.init.zeros_(blk.attn.c_proj.weight)   # zero-init (nanochat)
            nn.init.uniform_(blk.mlp.c_fc.weight, -s, s)
            nn.init.zeros_(blk.mlp.c_proj.weight)    # zero-init (nanochat)

    # ── Forward pass ──────────────────────────────────────────────────────────
    def forward(
        self,
        tokens:  torch.Tensor,                   # (B, T)  int64
        targets: Optional[torch.Tensor] = None,  # (B, T)  int64
    ) -> torch.Tensor:
        B, T = tokens.shape

        # Embed tokens + initial norm; save x0 for skip connections
        x  = self.wte(tokens).to(torch.bfloat16)  # (B, T, n_embd)
        x  = rms_norm(x)
        x0 = x                                    # skip-connection anchor

        # Trim rotary buffers to actual T
        cos = self.rot_cos[:, :T]   # (1, T, 1, head_dim//2)
        sin = self.rot_sin[:, :T]

        # Transformer blocks with per-layer residual routing
        for i, blk in enumerate(self.blocks):
            delta = blk(x, cos, sin)                         # (B, T, n_embd)
            x = self.resid_lambdas[i] * x + self.x0_lambdas[i] * x0 + delta

        # Final norm + unembed
        x      = rms_norm(x)
        logits = self.lm_head(x).float()          # (B, T, V)  fp32 for loss

        # Logit soft-cap  — prevents blow-up without clipping grad flow
        # nanochat uses 15; we use 30 for slightly wider dynamic range
        logits = 30.0 * torch.tanh(logits / 30.0)

        if targets is None:
            return logits

        # Chunked cross-entropy avoids materialising full (B*T, V) fp32 tensor
        loss = F.cross_entropy(
            logits.view(-1, self.vocab_size),
            targets.reshape(-1),
        )
        return loss

    # ── Parameter accounting ──────────────────────────────────────────────────
    def num_params(self) -> dict:
        """Like nanochat's num_scaling_params() — separates compute vs embedding."""
        wte_p  = self.wte.weight.numel()
        head_p = self.lm_head.weight.numel()
        sc_p   = self.resid_lambdas.numel() + self.x0_lambdas.numel()
        mat_p  = sum(p.numel() for blk in self.blocks for p in blk.parameters())
        total  = wte_p + head_p + sc_p + mat_p
        return dict(
            wte=wte_p, lm_head=head_p,
            transformer_matrices=mat_p,
            scalars=sc_p, total=total
        )

    def estimate_flops_per_token(self) -> int:
        """6N approximation (fwd + bwd) on compute-active params."""
        p = self.num_params()
        return 6 * (p['transformer_matrices'] + p['wte'])


# ── Quick sanity-check ────────────────────────────────────────────────────────
_model_cpu = GPT()
_p         = _model_cpu.num_params()
print('\nParameter breakdown:')
for k, v in _p.items():
    print(f'  {k:<25s}: {v/1e6:7.2f} M')
print(f'\n  FLOPs / token (6N): {_model_cpu.estimate_flops_per_token()/1e9:.3f} GFLOPs')
del _model_cpu



Parameter breakdown:
  wte                      :   38.63 M
  lm_head                  :   38.63 M
  transformer_matrices     :   84.93 M
  scalars                  :    0.00 M
  total                    :  162.20 M

  FLOPs / token (6N): 0.741 GFLOPs


In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Muon Optimizer  (Newton-Schulz orthogonalization)           ║
# ║  nanochat optim.py — adapted for XLA                         ║
# ╚══════════════════════════════════════════════════════════════╝

@torch.no_grad()
def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """
    Newton-Schulz iteration to compute G × (GᵀG)^{-1/2}  (i.e. the nearest
    orthogonal matrix to G).  Quintic polynomial schedule from nanochat.
    Runs natively in bfloat16 — fine on XLA / TPU.
    """
    assert G.ndim >= 2
    a, b, c = (3.4445, -4.7750, 2.0315)    # quintic poly coefficients
    X = G.to(dtype=torch.bfloat16)
    if X.size(-2) > X.size(-1):            # always work on 'fat' matrix
        X = X.mT
    # Spectral normalise to ≤ 1 to ensure convergence
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        A = X @ X.mT                       # XᵀX
        B = b * A + c * (A @ A)            # quintic correction
        X = a * X + B @ X
    if G.size(-2) > G.size(-1):
        X = X.mT
    return X


class Muon(torch.optim.Optimizer):
    """
    Muon — MomentUm Orthogonalized by Newton-schulz.
    Adapted from nanochat/optim.py.  Used for all weight *matrices*
    in the transformer blocks (not embeddings or scalars).

    Cautious weight decay: WD applied only where sign(grad) == sign(weight),
    so regularisation never fights the gradient direction.
    """

    def __init__(self, params, lr: float = MATRIX_LR,
                 momentum: float = MUON_MOMENTUM,
                 ns_steps: int   = MUON_NS_STEPS,
                 weight_decay: float = WEIGHT_DECAY):
        defaults = dict(lr=lr, momentum=momentum,
                        ns_steps=ns_steps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr       = group['lr']
            momentum = group['momentum']
            ns_steps = group['ns_steps']
            wd       = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad

                # Nesterov momentum
                state = self.state[p]
                if 'buf' not in state:
                    state['buf'] = torch.zeros_like(g)
                buf = state['buf']
                # buf = momentum * buf + (1 - momentum) * g
                buf.mul_(momentum).add_(g, alpha=(1.0 - momentum))
                g = g.mul(momentum).add_(buf)   # Nesterov update

                # Orthogonalise via Newton-Schulz
                g = zeropower_via_newtonschulz5(g, steps=ns_steps)

                # Scale: max(1, m/n)^0.5  (nanochat convention)
                scale = max(1.0, p.size(-2) / p.size(-1)) ** 0.5

                # Cautious weight decay (only where sign matches)
                if wd > 0:
                    # cast to fp32 for sign comparison
                    mask = (p.float() * g.float() > 0).to(p.dtype)
                    p.mul_(1.0 - lr * wd * mask)

                p.add_(g, alpha=-lr * scale)


# ── LR schedule helper ────────────────────────────────────────────────────────
def get_lr_factor(step: int) -> float:
    """
    Trapezoidal schedule used in nanochat:
    • 0 … WARMDOWN_START-1 : constant 1.0
    • WARMDOWN_START … NUM_STEPS : linear 1.0 → 0.0
    """
    if step < WARMDOWN_START:
        return 1.0
    progress = (step - WARMDOWN_START) / max(1, NUM_STEPS - WARMDOWN_START)
    return 1.0 - progress


def update_lr(step: int, muon: Muon, adamw: torch.optim.AdamW):
    """Update all param-group LRs according to the schedule."""
    f = get_lr_factor(step)
    for pg in muon.param_groups:
        pg['lr'] = f * MATRIX_LR
    for pg in adamw.param_groups:
        pg['lr'] = f * pg['_base_lr']   # stored at init


def build_optimizers(model: GPT) -> tuple:
    """
    nanochat parameter split:
      • Muon  → all 2-D weight matrices inside transformer blocks
      • AdamW → wte, lm_head, resid_lambdas, x0_lambdas
    """
    # 2-D matrices (Muon)
    matrix_params = [
        p for blk in model.blocks
        for p in blk.parameters()
        if p.ndim >= 2
    ]
    # Scalars inside blocks (should be none right now; future-proof)
    scalar_block_params = [
        p for blk in model.blocks
        for p in blk.parameters()
        if p.ndim < 2
    ]

    muon = Muon(
        matrix_params,
        lr=MATRIX_LR,
        momentum=MUON_MOMENTUM,
        ns_steps=MUON_NS_STEPS,
        weight_decay=WEIGHT_DECAY,
    )

    # AdamW parameter groups with individual base LRs
    adam_groups = [
        {'params': list(model.wte.parameters()),     '_base_lr': EMBED_LR,   'lr': EMBED_LR},
        {'params': list(model.lm_head.parameters()), '_base_lr': LMHEAD_LR,  'lr': LMHEAD_LR},
        {'params': [model.resid_lambdas],            '_base_lr': SCALAR_LR,  'lr': SCALAR_LR},
        {'params': [model.x0_lambdas],               '_base_lr': X0_LR,      'lr': X0_LR},
    ]
    if scalar_block_params:
        adam_groups.append({'params': scalar_block_params, '_base_lr': SCALAR_LR, 'lr': SCALAR_LR})

    adamw = torch.optim.AdamW(
        adam_groups,
        betas=ADAM_BETAS,
        eps=ADAM_EPS,
        weight_decay=0.0,   # WD handled in Muon; AdamW gets none
    )

    total_muon  = sum(p.numel() for p in matrix_params)
    total_adam  = sum(p.numel() for g in adam_groups for p in g['params'])
    print(f'Muon params  : {total_muon/1e6:.1f} M')
    print(f'AdamW params : {total_adam/1e6:.1f} M')
    return muon, adamw


print('Optimizer definitions ready.')


Optimizer definitions ready.


In [13]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Data Pipeline  — FineWeb-Edu + tiktoken, streamed           ║
# ╚══════════════════════════════════════════════════════════════╝
# Uses tiktoken r50k_base (GPT-2 tokenizer, vocab=50 257).
# Streams FineWeb-edu from HuggingFace, packs tokens into
# contiguous (B, T+1) slices, no padding needed.

import tiktoken

ENC = tiktoken.get_encoding('r50k_base')
EOT = ENC.eot_token   # 50 256

assert ENC.n_vocab <= VOCAB_SIZE, 'vocab mismatch'
print(f'Tokenizer: r50k_base  vocab={ENC.n_vocab}  padded_to={VOCAB_SIZE}  EOT={EOT}')


def tokenize_doc(text: str) -> list[int]:
    """Tokenize one document; append EOT separator."""
    return ENC.encode_ordinary(text) + [EOT]


def fineweb_token_stream(
    split_name: str = 'train',
    hf_name: str = 'sample-10BT',
    skip_first: int = 0,         # skip N tokens (e.g. val held-out)
    max_tokens: int | None = None,
) -> Iterator[int]:
    """
    Yields individual token IDs from FineWeb-Edu (streaming).
    Set skip_first > 0 to hold out a validation portion.
    """
    ds = load_dataset(
        'HuggingFaceFW/fineweb-edu',
        name=hf_name,
        split=split_name,
        streaming=True,
    )
    seen = 0
    emitted = 0
    for example in ds:
        toks = tokenize_doc(example['text'])
        for tok in toks:
            if seen < skip_first:
                seen += 1
                continue
            yield tok
            emitted += 1
            if max_tokens is not None and emitted >= max_tokens:
                return


# Pre-collect a validation shard  (10 M tokens, held out)
VAL_TOKENS = 10_000_000

def build_val_set() -> torch.Tensor:
    """
    Materialise 10 M tokens from the start of the stream as a
    flat CPU int16 tensor (saves RAM; values ≤ 50 304 fit in int16).
    """
    print(f'Building val set ({VAL_TOKENS/1e6:.0f} M tokens) …', flush=True)
    buf = []
    for tok in fineweb_token_stream(max_tokens=VAL_TOKENS):
        buf.append(tok)
    t = torch.tensor(buf, dtype=torch.int32)  # int32 to avoid overflow
    print(f'Val set built: {t.numel():,} tokens')
    return t


class TokenBatcher:
    """
    Streams tokens and yields (x, y) batches of shape
    (TOTAL_SEQS, SEQ_LEN) on CPU.  Runs in a background thread
    to overlap data prep with TPU compute.
    """

    def __init__(
        self,
        total_seqs: int = TOTAL_SEQS,
        seq_len: int    = SEQ_LEN,
        skip_first: int = VAL_TOKENS,
        prefetch: int   = 4,
    ):
        self.total_seqs = total_seqs
        self.seq_len    = seq_len
        self.tokens_per_batch = total_seqs * (seq_len + 1)
        self._q = queue.Queue(maxsize=prefetch)
        self._thread = threading.Thread(
            target=self._fill_queue,
            kwargs=dict(skip_first=skip_first),
            daemon=True,
        )
        self._thread.start()

    def _fill_queue(self, skip_first: int):
        buf = []
        n   = self.tokens_per_batch
        for tok in fineweb_token_stream(skip_first=skip_first):
            buf.append(tok)
            if len(buf) == n:
                t   = torch.tensor(buf, dtype=torch.int32)
                t   = t.view(self.total_seqs, self.seq_len + 1)
                x, y = t[:, :-1].long(), t[:, 1:].long()
                self._q.put((x, y))
                buf = []

    def __next__(self):
        return self._q.get()

    def __iter__(self):
        return self


print('Data pipeline definitions ready.')
print(f'Tokens per global batch: {TOTAL_SEQS*(SEQ_LEN+1):,}')
print(f'NOTE: streaming will begin when training starts.')
print(f'TIP:  set HF_DATASETS_CACHE env-var to a GCS bucket for re-use.')


Tokenizer: r50k_base  vocab=50257  padded_to=50304  EOT=50256
Data pipeline definitions ready.
Tokens per global batch: 65,600
NOTE: streaming will begin when training starts.
TIP:  set HF_DATASETS_CACHE env-var to a GCS bucket for re-use.


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SPMD sharding helpers & validation BPB                      ║
# ╚══════════════════════════════════════════════════════════════╝

def shard_batch(
    x: torch.Tensor,   # (B, T)  on CPU
    y: torch.Tensor,   # (B, T)  on CPU
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Move batch to XLA and annotate the batch dimension as data-parallel.
    Each of the 8 chips will receive 1/8 of the rows automatically.
    """
    x = x.to(DEVICE)
    y = y.to(DEVICE)
    # ('data', None)  →  shard dim-0 across the 8-chip 'data' axis;
    # leave dim-1 (seq) unsharded
    xs.mark_sharding(x, MESH, ('data', None))
    xs.mark_sharding(y, MESH, ('data', None))
    return x, y


@torch.no_grad()
def compute_val_bpb(
    model:    GPT,
    val_data: torch.Tensor,   # flat int32 tensor on CPU
    n_seqs:   int = 128,      # sequences to average over (quick estimate)
    seq_len:  int = SEQ_LEN,
) -> float:
    """
    Bits-per-byte on val_data.  Like nanochat's loss_eval.py —
    vocab-size-independent metric (useful across different tokenizers).

    bpb = (cross_entropy_nats / log(2)) / avg_bytes_per_token
    For tiktoken r50k, avg_bytes_per_token ≈ 4.0.
    """
    BYTES_PER_TOKEN = 4.0   # rough for GPT-2 BPE on English

    model.eval()
    losses = []
    stride  = n_seqs * (seq_len + 1)
    data    = val_data[:stride]

    chunk = data.view(n_seqs, seq_len + 1).long()
    xv, yv = chunk[:, :-1], chunk[:, 1:]
    xv, yv = shard_batch(xv, yv)

    loss = model(xv, yv)
    xm.mark_step()   # flush XLA graph and get scalar
    loss_val = loss.item()
    bpb = (loss_val / math.log(2)) / BYTES_PER_TOKEN

    model.train()
    return bpb


def grad_norm(model: GPT) -> float:
    """Global gradient L2 norm (for monitoring)."""
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.detach().float().norm().item() ** 2
    return total ** 0.5


print('SPMD helpers ready.')


SPMD helpers ready.


In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Main Training Loop                                          ║
# ╚══════════════════════════════════════════════════════════════╝

def pretrain(
    resume_step: int  = 0,    # set > 0 to resume from checkpoint
    max_steps:   int  = NUM_STEPS,
):
    # ── Build model on XLA ─────────────────────────────────────────────────
    model = GPT().to(DEVICE)
    xm.mark_step()   # let XLA materialise the empty parameter tensors

    p = model.num_params()
    flops_pt = model.estimate_flops_per_token()
    print(f'Model total   : {p["total"]/1e6:.1f} M params')
    print(f'  — wte       : {p["wte"]/1e6:.1f} M')
    print(f'  — lm_head   : {p["lm_head"]/1e6:.1f} M')
    print(f'  — matrices  : {p["transformer_matrices"]/1e6:.1f} M  ← N for scaling law')
    print(f'FLOPs/token  : {flops_pt/1e9:.3f} G  (6N approx)')
    print(f'Total train FLOPs: ~{flops_pt*TARGET_TOKENS:.3e}')

    # ── Optimizers ────────────────────────────────────────────────────────
    muon, adamw = build_optimizers(model)

    # ── Optionally resume ─────────────────────────────────────────────────
    start_step = 0
    if resume_step > 0:
        ckpt_path = f'{CKPT_DIR}/step_{resume_step:06d}.pt'
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu')
            model.load_state_dict(ckpt['model'], strict=True)
            muon.load_state_dict(ckpt['muon'])
            adamw.load_state_dict(ckpt['adamw'])
            start_step = resume_step
            print(f'Resumed from step {start_step}')
        else:
            print(f'WARNING: checkpoint {ckpt_path} not found, starting fresh.')

    # ── Data ──────────────────────────────────────────────────────────────
    print('\nBuilding validation set …')
    val_data = build_val_set()       # 10 M tokens, CPU

    print('\nStarting training data stream …')
    batcher = TokenBatcher(skip_first=VAL_TOKENS)  # skip val tokens

    # ── wandb (optional) ──────────────────────────────────────────────────
    if USE_WANDB:
        import wandb
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN,
            config=dict(
                depth=DEPTH, n_embd=N_EMBD, n_head=N_HEAD,
                seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE,
                total_tokens=TARGET_TOKENS, batch_tokens=TOTAL_BATCH_TOKENS,
                total_steps=NUM_STEPS, num_devices=NUM_DEVICES,
                matrix_lr=MATRIX_LR, muon_momentum=MUON_MOMENTUM,
                weight_decay=WEIGHT_DECAY, warmdown_start=WARMDOWN_START,
            ),
        )

    # ── Training ──────────────────────────────────────────────────────────
    model.train()
    t0 = time.perf_counter()
    step0_time = t0

    print(f'\n{'Step':>6}  {'loss':>8}  {'val_bpb':>9}  '
          f'{'tok/s':>9}  {'MFU%':>6}  lr_scale')
    print('-' * 65)

    # TPU peak TFLOPs/chip for v5e (bf16 matmuls)
    TPU_V5E_TFLOPS_PER_CHIP = 197.0   # bf16, single-chip peak
    PEAK_TFLOPS = TPU_V5E_TFLOPS_PER_CHIP * NUM_DEVICES * 1e12

    for step in range(start_step, max_steps):

        # ── LR schedule ──────────────────────────────────────────────────
        update_lr(step, muon, adamw)
        lr_f = get_lr_factor(step)

        # ── Get next batch ────────────────────────────────────────────────
        x_cpu, y_cpu = next(batcher)    # (TOTAL_SEQS, SEQ_LEN)  int64 CPU
        x, y = shard_batch(x_cpu, y_cpu)

        # ── Forward + backward ────────────────────────────────────────────
        muon.zero_grad(set_to_none=True)
        adamw.zero_grad(set_to_none=True)

        with torch.autocast(device_type='xla', dtype=torch.bfloat16):
            loss = model(x, y)

        loss.backward()

        # ── Gradient clip (optional; helps early training) ─────────────
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # ── Optimizer step + XLA graph commit ─────────────────────────
        # xm.optimizer_step handles gradient all-reduce for SPMD data-parallel
        # and calls xm.mark_step() internally
        xm.optimizer_step(muon,  barrier=False)
        xm.optimizer_step(adamw, barrier=False)
        xm.mark_step()   # commit the full step graph to XLA

        # ── Logging ───────────────────────────────────────────────────
        if step % LOG_EVERY == 0:
            t1 = time.perf_counter()
            dt = t1 - t0
            t0 = t1

            loss_val = loss.item()    # implicitly syncs TPU

            # Throughput
            toks_per_sec = TOTAL_BATCH_TOKENS * LOG_EVERY / dt
            # MFU = actual_flops / theoretical_peak_flops
            actual_tflops = toks_per_sec * flops_pt / 1e12
            mfu = 100.0 * actual_tflops / (PEAK_TFLOPS / 1e12)

            # Tokens seen so far
            tokens_seen = (step + 1) * TOTAL_BATCH_TOKENS

            print(f'{step:>6d}  {loss_val:>8.4f}  '
                  f'{" ":>9}  '
                  f'{toks_per_sec/1e6:>7.3f}M  '
                  f'{mfu:>5.1f}%  '
                  f'{lr_f:.3f}')

            if USE_WANDB:
                import wandb
                wandb.log(dict(
                    step=step, loss=loss_val,
                    tok_per_sec=toks_per_sec,
                    mfu=mfu, lr_factor=lr_f,
                    tokens_seen=tokens_seen,
                ), step=step)

        # ── Validation ────────────────────────────────────────────────
        if step % VAL_EVERY == 0 and step > 0:
            bpb = compute_val_bpb(model, val_data)
            print(f'{step:>6d}  {"":>8}  {bpb:>9.5f}  '
                  f'{" ":>9}  {" ":>6}  {"val"}')
            if USE_WANDB:
                import wandb
                wandb.log({'val_bpb': bpb, 'step': step}, step=step)

        # ── Checkpoint ────────────────────────────────────────────────
        if step % CKPT_EVERY == 0 and step > 0:
            ckpt_path = f'{CKPT_DIR}/step_{step:06d}.pt'
            # Pull weights to CPU before saving
            cpu_sd = {k: v.cpu() for k, v in model.state_dict().items()}
            torch.save({
                'step':  step,
                'model': cpu_sd,
                'muon':  muon.state_dict(),
                'adamw': adamw.state_dict(),
                'config': dict(
                    n_layer=N_LAYER, n_embd=N_EMBD, n_head=N_HEAD,
                    n_kv_head=N_KV_HEAD, head_dim=HEAD_DIM,
                    seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE,
                ),
            }, ckpt_path)
            print(f'  [ckpt] saved {ckpt_path}')

    # ── Final validation ──────────────────────────────────────────────────
    final_bpb = compute_val_bpb(model, val_data, n_seqs=512)
    total_time = time.perf_counter() - step0_time
    print(f'\n═══ Training complete ═══')
    print(f'Total time   : {total_time/3600:.2f} h')
    print(f'Final val bpb: {final_bpb:.5f}')
    print(f'Tokens trained: {max_steps*TOTAL_BATCH_TOKENS/1e9:.2f} B')

    # Final checkpoint
    cpu_sd = {k: v.cpu() for k, v in model.state_dict().items()}
    torch.save({'step': max_steps, 'model': cpu_sd,
                'muon': muon.state_dict(), 'adamw': adamw.state_dict()},
               f'{CKPT_DIR}/final.pt')
    print(f'Final checkpoint: {CKPT_DIR}/final.pt')

    if USE_WANDB:
        import wandb
        wandb.finish()

    return model


print('pretrain() defined.  Next cell: launch training.')


pretrain() defined.  Next cell: launch training.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  LAUNCH  — runs full Chinchilla-optimal pretraining          ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Estimated wall-clock on TPU v5e-8:
#   ~25-40 min depending on data-fetch latency and XLA compilation.
#   The first few steps are slow (XLA JIT tracing); steady-state
#   throughput kicks in after ~50 steps.
#
# To resume a run:
#   model = pretrain(resume_step=1000)

DEVICE = torch.device('xla')

model = pretrain()

/tmp/ipykernel_108/1623863673.py:11: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # let XLA materialise the empty parameter tensors


Model total   : 162.2 M params
  — wte       : 38.6 M
  — lm_head   : 38.6 M
  — matrices  : 84.9 M  ← N for scaling law
FLOPs/token  : 0.741 G  (6N approx)
Total train FLOPs: ~1.839e+18
Muon params  : 84.9 M
AdamW params : 77.3 M

Building validation set …
Building val set (10 M tokens) …


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Val set built: 10,000,000 tokens

Starting training data stream …

  Step      loss    val_bpb      tok/s    MFU%  lr_scale
-----------------------------------------------------------------


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/tmp/ipykernel_108/1623863673.py:102: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # commit the full step graph to XLA


     0   10.8268               0.017M    0.8%  1.000
    20    6.7223               0.030M    1.4%  1.000
    40    6.4281               0.305M   14.4%  1.000
    60    6.0915               0.303M   14.3%  1.000
    80    5.8269               0.304M   14.3%  1.000
   100    5.7232               0.306M   14.4%  1.000
   120    5.6881               0.304M   14.3%  1.000
   140    5.4127               0.300M   14.1%  1.000
   160    5.2774               0.305M   14.3%  1.000
   180    5.2309               0.307M   14.4%  1.000
   200    5.0604               0.305M   14.4%  1.000


/tmp/ipykernel_108/3352677584.py:48: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # flush XLA graph and get scalar


   200              1.89837                     val
   220    5.1321               0.144M    6.8%  1.000
   240    5.0981               0.303M   14.2%  1.000
   260    5.1560               0.300M   14.1%  1.000
   280    5.1132               0.301M   14.2%  1.000
   300    4.8545               0.299M   14.1%  1.000
   320    4.9855               0.303M   14.2%  1.000
   340    4.8216               0.028M    1.3%  1.000
   360    4.7594               0.294M   13.8%  1.000
   380    4.7465               0.299M   14.1%  1.000
   400    4.7553               0.303M   14.2%  1.000
   400              1.70786                     val
   420    4.6631               0.288M   13.6%  1.000
   440    4.5824               0.301M   14.2%  1.000
   460    4.6359               0.301M   14.2%  1.000
   480    4.4289               0.302M   14.2%  1.000
   500    4.4884               0.299M   14.1%  1.000
   520    4.3752               0.300M   14.1%  1.000
   540    4.2951               0.304M   14.3%  1

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Sampling (greedy / top-k / temperature)                     ║
# ╚══════════════════════════════════════════════════════════════╝

@torch.no_grad()
def sample(
    model:       GPT,
    prompt:      str,
    max_new:     int   = 200,
    temperature: float = 0.8,
    top_k:       int   = 50,
) -> str:
    model.eval()
    ids = torch.tensor([ENC.encode_ordinary(prompt)], dtype=torch.long).to(DEVICE)

    for _ in range(max_new):
        # Crop to model's max seq length
        ids_cond = ids[:, -SEQ_LEN:]
        logits   = model(ids_cond)          # (1, T, V)
        logits   = logits[:, -1, :] / temperature   # last token

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        probs    = torch.softmax(logits, dim=-1)
        xm.mark_step()   # materialise before multinomial
        next_id  = torch.multinomial(probs.cpu().float(), num_samples=1).to(DEVICE)

        if next_id.item() == EOT:
            break
        ids = torch.cat([ids, next_id], dim=1)

    return ENC.decode(ids[0].tolist())


# Test generation after training
prompts = [
    'The capital of France is',
    'In 1969, humans first',
    'The chemical formula for water is',
    'Machine learning is a field of',
]

print('=== Samples from trained model ===\n')
for p in prompts:
    out = sample(model, p, max_new=80, temperature=0.8)
    print(f'>>> {p}')
    print(out)
    print()


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Load & evaluate any saved checkpoint                        ║
# ╚══════════════════════════════════════════════════════════════╝

def load_checkpoint(path: str) -> GPT:
    ckpt  = torch.load(path, map_location='cpu')
    cfg   = ckpt.get('config', {})
    m     = GPT(**{k: v for k, v in cfg.items() if k in GPT.__init__.__code__.co_varnames})
    m.load_state_dict(ckpt['model'], strict=True)
    m     = m.to(DEVICE)
    xm.mark_step()
    print(f'Loaded checkpoint from step {ckpt.get("step", "?")}  →  {path}')
    return m


# Example usage:
#   m = load_checkpoint('./checkpoints/final.pt')
#   bpb = compute_val_bpb(m, val_data, n_seqs=512)
#   print(f'val bpb = {bpb:.5f}')

print('load_checkpoint() defined.')
print('Run: m = load_checkpoint("./checkpoints/final.pt")')


## Notes, Tuning Tips & Next Steps

### Architecture features from nanochat (all included)
| Feature | Source | Effect |
|---------|--------|--------|
| RoPE rotary embeddings | nanochat | Better length generalisation |
| QK-normalization | nanochat | Training stability, prevents attn entropy collapse |
| ReLU² activation | nanochat/modded-nanogpt | Cleaner sparsity than GELU |
| Per-layer λ_resid / λ_x0 scalars | nanochat | +0.003–0.01 bpb improvement |
| Logit soft-cap 30·tanh(·/30) | nanochat | Prevents logit blow-up |
| Untied wte / lm_head | nanochat | Independent embedding optimisation |
| Zero-init projections | nanochat | Better early-training dynamics |
| Muon + AdamW split | nanochat | Muon ≫ AdamW for matrices |
| Trapezoidal LR (0 warmup, 40% WD) | nanochat | Best schedule for fixed budget |

### Value Embeddings (not included — adds ~231 M extra params at depth-12)
nanochat uses per-layer token-level value embeddings at alternating layers.  
Add them back in for **larger models** (depth ≥ 20) where the extra capacity  
is meaningful without doubling total parameter count.

### Scaling to larger depth
```python
DEPTH = 20  # → n_embd=1280, ~560 M total params, ~10 B Chinchilla tokens
DEPTH = 24  # → n_embd=1536, GPT-2 1.5B-class, ~3 hr on v5e-8
```

### GCS checkpointing
```bash
export CKPT_DIR = 'gs://your-bucket/nanochat-runs/d12'
```
torch.save / torch.load work transparently with gs:// paths via gcsfs.

### FP8 training (TPU v5e supports it)
nanochat uses FP8 via transformer-engine on H100. For TPU v5e, XLA's  
built-in BF16 is already ~optimal; FP8 support in torch_xla is experimental.

### SFT after pretraining
Load `./checkpoints/final.pt`, clone nanochat and run:  
```bash
python -m scripts.chat_sft --resume-from ./checkpoints/final.pt
```
